# 02 · pychameleon vs Moonpuck — jakość, szybkość, skalowalność

Cel: pokazać że nasza implementacja jest **porównywalna jakościowo** a **wielokrotnie szybsza** od oryginalnej Moonpuck.

Dane:
- `results/comparison.csv` — head-to-head na 3 wspólnych datasetach (aggregation, smileface, t4_8k).
- `benchmarks/reference_moonpuck/*/meta.json` — runtime Moonpucka (ground-truth dla speedup).
- `results/scalability.csv` — skalowalność `runtime(n)` i `runtime(d)` na syntetycznych `make_blobs`.

In [ ]:
import sys, os, importlib
sys.path.insert(0, os.getcwd())
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import _helpers as H
importlib.reload(H)  # w razie gdyby kernel zachowal stara wersje modulu

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120
pd.options.display.float_format = '{:.3f}'.format

## 1. Tabela head-to-head

3 datasety na których Moonpuck był uruchomiony. `runtime_moonpuck_s` pochodzi z `meta.json` benchmarków (Moonpuck implementation, ten sam zestaw parametrów co publikowany w paperze).

In [ ]:
comp = H.load_comparison().set_index('dataset')
h2h = comp.loc[H.DATASETS_WITH_MOONPUCK,
               ['n', 'runtime_s', 'runtime_moonpuck_s', 'ari_vs_moonpuck']].copy()
h2h['speedup'] = h2h['runtime_moonpuck_s'] / h2h['runtime_s']
h2h = h2h.rename(columns={'runtime_s': 'pychameleon [s]',
                            'runtime_moonpuck_s': 'Moonpuck [s]',
                            'ari_vs_moonpuck': 'ARI(ours, Moonpuck)'})
h2h = h2h[['n', 'pychameleon [s]', 'Moonpuck [s]', 'speedup', 'ARI(ours, Moonpuck)']]
h2h = h2h.sort_values('speedup')
h2h.style.format({'pychameleon [s]': '{:.3f}', 'Moonpuck [s]': '{:.2f}',
                  'ARI(ours, Moonpuck)': '{:.3f}', 'speedup': '{:.1f}×'})

**Konkluzja:** pychameleon jest **od kilkunastu do ~260× szybszy** od Moonpucka. Niska zgodność labelek między implementacjami (ARI 0.58–0.88) nie oznacza że jedna z nich się myli — sekcja 3 pokazuje że dla aggregation pychameleon trafia w prawdziwy GT znacznie celniej niż Moonpuck.

Skąd ten speedup:
- **cache na `_internal_bisector`** — internal bisection sub-grafu (najdroższa operacja) liczona raz na klaster i memoizowana,
- **lazy invalidation w heap-u merger'a** — zamiast rebuildować priority queue po każdym merge'u, oznaczamy entries jako nieaktualne i pomijamy przy pop'ie,
- **pymetis zamiast Pythonowego rekursywnego bisectora** — natywne METIS jest rzędy wielkości szybsze.

## 2. Speedup w funkcji rozmiaru datasetu

Każdy punkt to jeden dataset z head-to-head. Oś Y na skali log, bo speedup rozjeżdża się od ~16× do ~270×.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(h2h['n'], h2h['speedup'], s=80, color='#55A868', zorder=3)
for name, row in h2h.iterrows():
    ax.annotate(f"{H.PRETTY[name]}\n{row['speedup']:.1f}×",
                (row['n'], row['speedup']),
                xytext=(8, 0), textcoords='offset points',
                va='center', fontsize=9)
ax.set_yscale('log')
ax.set_xlabel('liczba punktów n')
ax.set_ylabel('speedup (Moonpuck / pychameleon)')
ax.set_title('Speedup vs rozmiar datasetu')
ax.axhline(1.0, ls=':', color='gray', linewidth=0.8)
ax.grid(True, which='both', linestyle=':', alpha=0.4)
ax.margins(x=0.18)
plt.tight_layout()
plt.show()

## 3. Galeria 3-kolumnowa — ground truth · labelki · błędy

Tu nie porównujemy obu implementacji ze sobą, tylko **obie do prawdziwego GT** (Aggregation: Gionis 2007 z UEF/Fränti; t4_8k: kanoniczne labelki CLUTO = nasz DS3). Dla każdego datasetu dwa wiersze (Moonpuck, pychameleon), trzy kolumny:

1. **Ground truth** — prawdziwe klastry (`-1` szum szaro).
2. **Labelki** — wynik danej implementacji, po hungarian remappingu na przestrzeń GT (kolory się zgadzają).
3. **Correctness** — zielony = zgodny z GT, czerwony = błąd; tytuł podaje accuracy.

Smileface nie ma publikowanego GT (custom synthetic), więc wypada z tej sekcji.

In [ ]:
# Dla t4_8k Moonpuck odniesienie jest pod 't4_8k', ale optymalne labelki pychameleona
# pochodzą z runu z parametrami HPO (DS3 = ten sam point cloud).
pychameleon_label_source = {
    'aggregation': 'aggregation',
    't4_8k': 'ds3',
}

galerie = [
    ('aggregation', 'aggregation'),
    ('t4_8k', 't4_8k'),
]

fig, axes = plt.subplots(2 * len(galerie), 3,
                          figsize=(11, 3.6 * len(galerie) * 2),
                          gridspec_kw={'hspace': 0.30, 'wspace': 0.05})

for i, (name, moon_key) in enumerate(galerie):
    gt_df = H.load_ground_truth(name)
    if gt_df is None:
        raise RuntimeError(
            f"load_ground_truth({name!r}) zwrocilo None — zrestartuj kernel."
        )
    gt_labels = gt_df['label'].values

    moon_df = H.load_moonpuck_labels(moon_key)
    pred_df = H.load_pychameleon_labels(pychameleon_label_source[name])

    moon_aligned = H.align_labels_to_gt(moon_df['label'].values, gt_labels)
    pred_aligned = H.align_labels_to_gt(pred_df['label'].values, gt_labels)
    moon_view = moon_df.copy(); moon_view['label'] = moon_aligned
    pred_view = pred_df.copy(); pred_view['label'] = pred_aligned

    for row_offset, impl, view, aligned in [
        (0, 'Moonpuck', moon_view, moon_aligned),
        (1, 'pychameleon', pred_view, pred_aligned),
    ]:
        r = 2 * i + row_offset
        H.scatter(axes[r, 0], gt_df, title=f'{H.PRETTY[name]} — ground truth')
        H.scatter(axes[r, 1], view, title=f'{H.PRETTY[name]} — {impl}')
        nc, ni, _ = H.correctness_scatter(
            axes[r, 2], gt_df[['x0', 'x1']], gt_labels, aligned, title='',
        )
        pct = 100 * nc / (nc + ni) if (nc + ni) else 0
        axes[r, 2].set_title(
            f'{H.PRETTY[name]}, accuracy {pct:.1f}%', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Skalowalność `runtime(n)` — log-log

`make_blobs`, `d = 2`, 5 izotropowych klastrów, 3 powtórzenia na każdy `n`. Oczekiwana asymptotyka algorytmu: **O(n log n)** dla budowy k-NN przez KDTree + **O(m² log m)** dla merger'a (m = liczba sub-klastrów po Fazie I, m ≪ n).

In [ ]:
scal = H.load_scalability()
scal_n = scal[scal['kind'] == 'n'].copy()
agg_n = scal_n.groupby('n')['runtime_s'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(agg_n['n'], agg_n['mean'], yerr=agg_n['std'],
            marker='o', linestyle='-', color='#4C72B0', capsize=3, label='pychameleon (mean ± std)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('liczba punktów n')
ax.set_ylabel('runtime [s]')
ax.set_title('Skalowalność: runtime(n), d=2')

# Linear fit on log-log to estimate empirical slope.
logn = np.log10(agg_n['n'])
logt = np.log10(agg_n['mean'])
slope, intercept = np.polyfit(logn, logt, 1)
xs = np.linspace(logn.min(), logn.max(), 50)
ax.plot(10**xs, 10**(slope*xs + intercept), '--', color='gray', linewidth=0.8,
        label=f'fit: runtime ∝ n^{slope:.2f}')
ax.legend()
ax.grid(True, which='both', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()
print(f'Empiryczny wykładnik: {slope:.2f}  (teoretycznie ~1.0–1.3)')

## 5. Wnioski

1. **Speedup vs Moonpuck**: 16× (smileface) – **~260× (t4.8k)**. Implementacja jest produkcyjnie szybsza.
2. **Jakość vs prawdziwy GT** (sekcja 3): pychameleon trafia w GT lepiej niż Moonpuck na obu datasetach — Aggregation 97.8% vs 71.2%, t4_8k 77.3% vs 67.9%. t4_8k ma cap accuracy ~80% bo zawiera ~10% szumu w GT, a goły CHAMELEON nie ma noise-handlingu (znana granica algorytmu, nie błąd implementacji).
3. **Empiryczna skalowalność `runtime(n)`** ma wykładnik bliski **liniowo-logarytmicznemu** — zgodnie z teoretyczną predykcją O(n log n + m² log m).
4. **Ograniczenie wymiarowości**: KDTree przestaje być efektywne dla `d > 10` (curse of dimensionality). W obecnym scope projektu (2D shape clustering z papera Karypis) to nie problem. Szczegóły w `docs/documentation/05_efektywnosc.md`.